# Tulu Mitra — fine-tune Whisper on Tulu recordings (LoRA)

Runs the three scripts in `training/` on a free Colab GPU.

**Before you start**
1. Upload the whole `tulu-mitra-backend` folder to Google Drive (e.g. `MyDrive/tulu-mitra-backend`).
2. Put recordings in `training/recordings/` named like `Ab_001.wav` (speaker + sentence number from `training/recording_script.csv`; any audio format). Check the `tulu` column of the script first.
3. **Runtime → Change runtime type → T4 GPU.**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Drive and open the project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/tulu-mitra-backend'  # ← change if you put it elsewhere
%cd $PROJECT
!ls training/recordings

## 2. Install training packages

In [ ]:
!pip install -q -r training/requirements-train.txt

## 3. Prepare the dataset
Converts to 16 kHz mono, trims silence, holds out 10% of sentences for testing. Use `--split speaker` instead to hold out a whole speaker (needs ≥ 2 speakers).

In [ ]:
!python training/prepare_dataset.py
import json; print(json.dumps(json.load(open('training/processed/report.json')), indent=2, ensure_ascii=False))

## 4. Train
whisper-small + LoRA (r=8, α=32, q_proj/v_proj). On a T4, an hour of audio × 10 epochs takes roughly 30–60 minutes. The script scores the untuned model first, trains, then scores the fine-tuned one.

In [ ]:
!python training/train_whisper_lora.py --epochs 10

## 5. Results for your report

In [ ]:
import json
m = json.load(open('models/whisper-tulu-lora/metrics.json'))
for k in ('baseline', 'fine_tuned'):
    if k in m: print(f"{k:11}  WER {m[k]['wer']:.1%}   CER {m[k]['cer']:.1%}   (n={m[k]['n']})")
print()
for ex in m['examples'][:8]:
    print('REF :', ex['reference']); print('PRED:', ex['prediction']); print()

## 6. Optional: per-clip predictions for error analysis
Writes `results/asr_predictions.csv`.

In [ ]:
!python training/evaluate_asr.py

## 7. Use the model in the backend
The adapter is already in `models/whisper-tulu-lora/` inside your Drive copy. Copy that folder into the backend you run (same path: `models/whisper-tulu-lora/`) and restart it. `GET /health` will then show `fine_tuned_adapter`.

To download it as a zip instead:

In [ ]:
!cd models && zip -qr /content/whisper-tulu-lora.zip whisper-tulu-lora
from google.colab import files; files.download('/content/whisper-tulu-lora.zip')